In [1]:
!pip install peft evaluate

In [2]:
import numpy as np
import pandas as pd
import torch

In [4]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cuda


**Some funny df from the internet**

In [5]:
df = pd.read_csv('/kaggle/input/datasets/dinayatsuk/woman-clothing/women-clothing-accessories.csv', sep='\t')
df

,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative
...,...,...
89995,сделано достаточно хорошо. на ткани сделан рис...,positive
89996,Накидка шикарная. Спасибо большое провдо линяе...,positive
89997,спасибо большое ) продовца рекомендую.. заказа...,positive
89998,Очень довольна заказом! Меньше месяца в РБ. К...,positive


In [6]:
df['sentiment'].unique()

array(['negative', 'neautral', 'positive'], dtype=object)

In [7]:
df['sentiment'].value_counts()

sentiment
negative    30000
neautral    30000
positive    30000
Name: count, dtype: int64

In [8]:
df.isna().value_counts()

review  sentiment
False   False        90000
Name: count, dtype: int64

In [9]:


df['review'] = df['review'].astype(str)
df['sentiment'] = df['sentiment'].replace({"neautral": "neutral"})

In [10]:
df['sentiment'].unique()

array(['negative', 'neutral', 'positive'], dtype=object)

In [11]:
longest_idx = df['review'].str.len().idxmax()

longest_string = df['review'].loc[longest_idx]
longest_string

'Качество шубы просто отвратительное. Вся шуба дырявая, швы пошиты как попало, черными нитками розовая шуба! Линяет ужасно! На розовой замше красными нитками нарисовано! Замшу видно! Крючки отваливаются, и пришиты тяп ляп. (Кстати уже отдала 1800 чтобы перешить крючки!) \r\nМеха мало и он жидкий, линяет ппц! Сроду такого позора мехового не видела!\r\nПродавец и размер посоветовал не верный, мне большая! Пришлось внахлест перешивать крючки!\r\nОткрыла спор, али не одобрил возврат 6000 на ремонт шубы, говорят отправьте обратно! Обратно? Серьёзно? И платить пересыл? Подала жалобу на возврат 6000, жду пересмотра спора! \r\nЦена шубе 8000 не больше! На один сезон шуба и то если носить раз в неделю, сыпется жутко,вся вылезет!\r\nПродавцу написала что шуба вся бракованная, молчит! Даже не извинился! И не предложил решить вопрос!\r\nЕсли али не одобрит жалобу я буду просто в шоке! Отпишусь тут по результатам. \r\nШУБУ НЕ СОВЕТУЮ! НЕ ТРАТЬТЕ ДЕНЬГИ! Прийдет лысая и драная! Лучше доплатить 5000 

In [12]:

shortest_indices = df['review'].str.len().nsmallest(10000).index
shortest_reviews = df.loc[shortest_indices]
print(shortest_reviews)


                                 review sentiment
1978                                  r  negative
3959                                  .  negative
4445                                  .  negative
5630                                  п  negative
5823                                  я  negative
...                                 ...       ...
15091  средненько не советую, рукав 3/4  negative
15095  просвечивает как москитная сетка  negative
15198  Заказ не пришёл, деньги вернули   negative
15199  Товар не пришёл, деньги вернули.  negative
15271  Товар не пришёл. Деньги вернули.  negative

[10000 rows x 2 columns]


In [13]:
# Удаляем отзывы, где меньше 3 символов, потому что как вообще?! это оценивать
df = df[df['review'].str.len() >= 3]

In [14]:
shortest_indices = df['review'].str.len().nsmallest(10000).index
shortest_reviews = df.loc[shortest_indices]
print(shortest_reviews)


                                 review sentiment
5277                                Дно  negative
8246                                ...  negative
8511                                !!!  negative
8985                                лдж  negative
15061                               ...  negative
...                                 ...       ...
35117  Пришло платье на размер больше.    neutral
35204  нормальная. но чистая синтетика.   neutral
35237  Товар не пришёл. Деньги вернули.   neutral
35256  только на маленькую грудь(((((((   neutral
35330  Very thin, certanly not 350 g/m2   neutral

[10000 rows x 2 columns]


In [15]:
df['sentiment'].value_counts()

sentiment
negative    29969
positive    29929
neutral     29915
Name: count, dtype: int64

**Encode lables and split on train & test**

In [16]:
label_names = df["sentiment"].unique()
label2id = {l: i for i, l in enumerate(label_names)}
id2label = {i: l for l, i in label2id.items()}
df.loc[:, "label_id"] = df["sentiment"].map(label2id)
print(label2id)
print(id2label)

{'negative': 0, 'neutral': 1, 'positive': 2}
{0: 'negative', 1: 'neutral', 2: 'positive'}


/tmp/ipykernel_1116/353898798.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, "label_id"] = df["sentiment"].map(label2id)


In [17]:
df.head()

,review,sentiment,label_id
0,качество плохое пошив ужасный (горловина напер...,negative,0
1,"Товар отдали другому человеку, я не получила п...",negative,0
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative,0
3,"товар не пришел, продавец продлил защиту без м...",negative,0
4,"Кофточка голая синтетика, носить не возможно.",negative,0


In [18]:
from sklearn.model_selection import train_test_split
from transformers import set_seed

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


In [19]:
SEED = 42
set_seed(SEED)

train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label_id"], random_state=42)
val_df, test_df = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label_id"], random_state=42)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

train=71850  val=8981  test=8982


**Tokenization**

In [20]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_id = "deepvk/RuModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id, attn_implementation="sdpa")


Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

In [21]:
probe = train_df['review'].sample(len(train_df), random_state=SEED).tolist()
lens = [len(x) for x in tokenizer(probe)["input_ids"]]
max_len = max(lens)
print(max_len)

1230


In [22]:
print("95th percentile:", np.percentile(lens, 95))
print("99th percentile:", np.percentile(lens, 99))
print("Fraction exceeding 512:", np.mean(np.array(lens) > 512))
print("Fraction exceeding 1024:", np.mean(np.array(lens) > 1024))

95th percentile: 103.0
99th percentile: 179.0
Fraction exceeding 512: 5.5671537926235215e-05
Fraction exceeding 1024: 1.3917884481558804e-05


In [23]:
from datasets import Dataset

def make_labels(ids):
    ids = np.asarray(ids)
    return ids.tolist()


def make_ds(dataframe):
    ds = Dataset.from_dict({
        "text": dataframe['review'].tolist(),
        "labels": make_labels(dataframe["label_id"].values),
    })
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=512),
        batched=True,
        remove_columns=["text"],
    )


train_ds, val_ds, test_ds = make_ds(train_df), make_ds(val_df), make_ds(test_df)
train_ds

Map:   0%|          | 0/71850 [00:00<?, ? examples/s]

Map:   0%|          | 0/8981 [00:00<?, ? examples/s]

Map:   0%|          | 0/8982 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 71850
})

**Training**

In [24]:
!pip install -U torchao

In [25]:
import os

import numpy as np
import torch
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from peft import LoraConfig, PeftModel, TaskType, get_peft_model

In [26]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.numel())

model.embeddings.tok_embeddings.weight 38682624
model.embeddings.norm.weight 768
model.layers.0.attn.Wqkv.weight 1769472
model.layers.0.attn.Wo.weight 589824
model.layers.0.mlp_norm.weight 768
model.layers.0.mlp.Wi.weight 1769472
model.layers.0.mlp.Wo.weight 884736
model.layers.1.attn_norm.weight 768
model.layers.1.attn.Wqkv.weight 1769472
model.layers.1.attn.Wo.weight 589824
model.layers.1.mlp_norm.weight 768
model.layers.1.mlp.Wi.weight 1769472
model.layers.1.mlp.Wo.weight 884736
model.layers.2.attn_norm.weight 768
model.layers.2.attn.Wqkv.weight 1769472
model.layers.2.attn.Wo.weight 589824
model.layers.2.mlp_norm.weight 768
model.layers.2.mlp.Wi.weight 1769472
model.layers.2.mlp.Wo.weight 884736
model.layers.3.attn_norm.weight 768
model.layers.3.attn.Wqkv.weight 1769472
model.layers.3.attn.Wo.weight 589824
model.layers.3.mlp_norm.weight 768
model.layers.3.mlp.Wi.weight 1769472
model.layers.3.mlp.Wo.weight 884736
model.layers.4.attn_norm.weight 768
model.layers.4.attn.Wqkv.weight 176

In [27]:
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
    attn_implementation="sdpa",
)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules='all-linear',
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,406,083 || all params: 153,013,254 || trainable%: 2.2260


In [28]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(
            labels, predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            labels, predictions,
            average="weighted",
            zero_division=0,
        ),
    }

In [29]:
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LR = 2e-4

training_args = TrainingArguments(
    output_dir="lora-sentiment",
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,

    learning_rate=LR,
    weight_decay=0.01,
    lr_scheduler_type="linear",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    bf16=True,
    label_names=["labels"],
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,  # новая модель с подключённой LoRA
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,4.124711,1.014201,0.768177,0.763258,0.763288
2,3.764471,0.973733,0.782764,0.781894,0.781914
3,3.312227,0.986240,0.782430,0.783780,0.783791


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=6738, training_loss=3.859590416599997, metrics={'train_runtime': 12814.9974, 'train_samples_per_second': 16.82, 'train_steps_per_second': 0.526, 'total_flos': 1.42357343378562e+16, 'train_loss': 3.859590416599997, 'epoch': 3.0})

In [33]:
import json
trainer.save_model("lora-sentiment")
tokenizer.save_pretrained("lora-sentiment")

('lora-sentiment/tokenizer_config.json', 'lora-sentiment/tokenizer.json')

In [41]:
import os
import json

os.makedirs("lora-sentiment", exist_ok=True)

with open(os.path.join("lora-sentiment", "labels.json"), "w", encoding="utf-8") as f:
    json.dump({
        "labels": label_names.tolist(),
        "id2label": {str(i): label for i, label in enumerate(label_names.tolist())},
        "label2id": {label: i for i, label in enumerate(label_names.tolist())},
        "base_model": "deepvk/RuModernBERT-base",
        "max_length": 512
    }, f, ensure_ascii=False, indent=2)

print("labels.json saved successfully")

labels.json saved successfully


In [34]:
test_metrics = trainer.evaluate(
    eval_dataset=test_ds,
    metric_key_prefix="test",
)
print(test_metrics)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
3.312227,0.995640,3,0.784124,0.785756,0.785753


{'test_loss': 0.9956399202346802, 'test_accuracy': 0.7841238031618794, 'test_f1_macro': 0.7857555444096717, 'test_f1_weighted': 0.7857533158387996}


In [42]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

MODEL_ID = "deepvk/RuModernBERT-base"
ADAPTER_DIR = "lora-sentiment"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
with open(f"{ADAPTER_DIR}/labels.json", encoding="utf-8") as f:
    config = json.load(f)

id2label = {int(k): v for k, v in config["id2label"].items()}
MAX_LENGTH = config["max_length"]

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(id2label),
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
    problem_type="single_label_classification",
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

model.to(device)
model.eval()

print("Device:", device)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


In [45]:
def predict_sentiment(text: str) -> dict:
    if not isinstance(text, str) or not text.strip() or len(text) < 3:
        raise ValueError("Передай непустой текст отзыва и не меньше 3-х символов.")

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    predicted_id = int(probabilities.argmax().item())

    return {
        "text": text,
        "sentiment": id2label[predicted_id],
        "confidence": round(float(probabilities[predicted_id]), 4),
        "probabilities": {
            id2label[i]: round(float(probabilities[i]), 4)
            for i in range(len(probabilities))
        },
    }

In [48]:
reviews = [
    "Качество ужасное, ткань тонкая, швы разошлись. Я недовольна",
    "Обычная вещь, качество среднее, ничего особенного.",
    "Отличное качество, всё подошло, покупкой очень довольна!",
    "Мрак, ужас, качество плохое"
]

for review in reviews:
    result = predict_sentiment(review)
    print(f'{result["sentiment"]} ({result["confidence"]:.2%}): {result["text"]}')

negative (88.73%): Качество ужасное, ткань тонкая, швы разошлись. Я недовольна
neutral (86.36%): Обычная вещь, качество среднее, ничего особенного.
positive (99.78%): Отличное качество, всё подошло, покупкой очень довольна!
negative (85.27%): Мрак, ужас, качество плохое
